# Slack Helper

This notebook shows examples of posting to a Slack channel via a Slack App.
The first method uses an [incoming webhook](https://docs.slack.dev/messaging/sending-messages-using-incoming-webhooks/), and a more advanced method is shown at the bottom (for more advanced formatting) that uses the Slack Web API.

### Storing and Retrieving the Webhook URL

In [0]:
webhook_url_temp = "https://hooks.slack.com/services/..." # Fill this out and then set this back to "" after running the Databricks Secrets cells below


In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

secret_scope = "social_listening_app"
workspace_client = WorkspaceClient()

# Create the secret scope if it doesn't already exist
try:
    workspace_client.secrets.create_scope(scope=secret_scope)
    print(f"Created secret scope: {secret_scope}")    
except ResourceAlreadyExists:
    print(f"Secret scope '{secret_scope}' already exists, continuing...")

# Add the secret
workspace_client.secrets.put_secret(scope=secret_scope, key="slack_webhook_url", string_value=webhook_url_temp)
print("Added Slack webhook URL")

In [0]:
webhook_url = dbutils.secrets.get(scope="social_listening_app", key="slack_webhook_url")

### Posting a Simple Message

In [0]:
import requests

response = requests.post(
    webhook_url,
    headers={"Content-type": "application/json"},
    json={"text": "Hello from Databricks!"}
)

print(response.status_code, response.text)


### Posting Persona Reports

In [0]:
reports_table = "my_catalog.my_schema.feedback_content_reports" # Replace with your catalog/schema name

In [0]:
# Helper functions

import re

def md_to_slack(text):
    """Convert markdown formatting to Slack mrkdwn."""
    # Convert ### headings to bold
    text = re.sub(r"^###\s*(.+)$", r"*\1*", text, flags=re.MULTILINE)
    # Convert ## headings to bold
    text = re.sub(r"^##\s*(.+)$", r"*\1*", text, flags=re.MULTILINE)
    # Convert **bold** to *bold*
    text = re.sub(r"\*\*(.+?)\*\*", r"*\1*", text)
    # Convert bullet dashes to bullet points
    text = re.sub(r"^- ", "• ", text, flags=re.MULTILINE)
    # Clean up any resulting double-bold (***text*** → *text*)
    text = re.sub(r"\*{2,}(.+?)\*{2,}", r"*\1*", text)
    return text.strip()

def split_text_into_chunks(text, max_chars=3000):
    """Split text into chunks at paragraph boundaries, respecting Slack's 3000-char limit."""
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        # If a single paragraph exceeds the limit, split it by lines
        if len(para) > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""
            lines = para.split("\n")
            for line in lines:
                if len(current_chunk) + len(line) + 1 > max_chars:
                    chunks.append(current_chunk.strip())
                    current_chunk = line + "\n"
                else:
                    current_chunk += line + "\n"
        elif len(current_chunk) + len(para) + 2 > max_chars:
            chunks.append(current_chunk.strip())
            current_chunk = para + "\n\n"
        else:
            current_chunk += para + "\n\n"

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

In [0]:
df = spark.table(reports_table)
games = [row.game_name for row in df.select("game_name").distinct().collect()]

controller_emoji = "\U0001f3ae"
person_emoji = "\U0001f464"

for game in sorted(games):
    game_reports = df.filter(df.game_name == game).select("persona", "report_contents").collect()

    # Game title — header block (largest text for webhooks)
    blocks = [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": f"{controller_emoji} {game}", "emoji": True}
        },
        {"type": "divider"}
    ]

    for row in game_reports:
        persona_label = row.persona.replace("_", " ").title()

        # Persona subtitle — bold mrkdwn section
        blocks.append({
            "type": "section",
            "text": {"type": "mrkdwn", "text": f"*{person_emoji} {persona_label}*"}
        })

        report_text = row.report_contents.strip() if row.report_contents else "No report available."
        formatted = md_to_slack(report_text)
        # Wrap in block quote by prefixing each line with >
        formatted = "\n".join(f">{line}" for line in formatted.split("\n"))
        chunks = split_text_into_chunks(formatted)

        for chunk in chunks:
            blocks.append({"type": "section", "text": {"type": "mrkdwn", "text": chunk}})

        blocks.append({"type": "divider"})

    # Slack limits 50 blocks per message — split into multiple messages if needed
    for i in range(0, len(blocks), 50):
        batch = blocks[i:i + 50]
        payload = {
            "text": f"Persona reports for {game}",
            "blocks": batch
        }
        response = requests.post(
            webhook_url,
            headers={"Content-type": "application/json"},
            json=payload
        )
        print(f"{game} (batch {i // 50 + 1}): {response.status_code} {response.text}")

## Advanced Formatting

This is achieved by using the Slack Web API instead of using a webhook. More information [here](https://docs.slack.dev/apis/web-api/).

You will need the following, which you can find by opening the [app settings](https://api.slack.com/apps) > OAuth and Permissions
1. Bot user OAuth token
    - Under "OAuth Tokens", will look like `xoxb-...`
1. Bot token scope: `chat:write`
    - Under "Scopes" section 
    - You may need to reinstall the app to the workspace after adding these scopes.

You will also need the Channel ID of channels you want to post to. These are not sensitive, and can be found by opening the channel in Slack and opening the channel info (e.g. click the channel name at the top of the page after opening it).

Add the bot to the channel by going to the channel in Slack and using `/invite`.


In [0]:
# Instead of a webhook URL, you need a bot/app token
bot_token_temp = "xoxb-..."
workspace_client.secrets.put_secret(scope=secret_scope, key="slack_bot_token", string_value=bot_token_temp)

In [0]:
import re
import time
import requests

# ── Slack config ─────────────────────────────────────────────────────────────
SLACK_BOT_TOKEN = dbutils.secrets.get("social_listening_app", "slack_bot_token")  # xoxb-… , chat:write
CHANNEL_ID      = "C0ABC123..."   # Not sensitive, get from Slack interface
POST_URL        = "https://slack.com/api/chat.postMessage"

# ── Slack markdown-block limits ──────────────────────────────────────────────
MAX_BLOCKS_PER_MSG = 50      # hard limit: blocks per message
MAX_MD_CHARS       = 11800   # safety margin under the 12,000-char cumulative markdown limit
MAX_CHUNK_CHARS    = 11000   # max size of a single markdown block

controller_emoji = "\U0001f3ae"
person_emoji     = "\U0001f464"


def split_markdown(text, max_chars=MAX_CHUNK_CHARS):
    """Split markdown into pieces <= max_chars at blank-line boundaries, never inside a code fence."""
    chunks, current, in_fence = [], "", False
    for para in text.split("\n\n"):
        candidate = f"{current}\n\n{para}" if current else para
        if not in_fence and current and len(candidate) > max_chars:
            chunks.append(current)
            current = para
        else:
            current = candidate
        if para.count("```") % 2 == 1:
            in_fence = not in_fence
    if current:
        chunks.append(current)

    safe = []
    for c in chunks:
        while len(c) > max_chars:
            cut = c.rfind("\n", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            safe.append(c[:cut])
            c = c[cut:].lstrip("\n")
        safe.append(c)
    return [c for c in safe if c.strip()]


def heading_level(line):
    """Markdown heading level (1-6) of a line, or None if it isn't a # heading."""
    m = re.match(r"^\s{0,3}(#{1,6})\s+\S", line)
    return len(m.group(1)) if m else None


def split_at_summary(report_text):
    """Return (head, rest): head = everything up to & including the Summary section;
    rest = the remainder. If no Summary heading (or no following section) is found, rest is ''."""
    lines = report_text.split("\n")
    summary_idx = summary_lvl = None
    for i, line in enumerate(lines):
        lvl = heading_level(line)
        if lvl is not None and "summary" in line.lower():
            summary_idx, summary_lvl = i, lvl
            break
    if summary_idx is None:
        return report_text, ""                        # no Summary heading -> keep it all in the parent
    for j in range(summary_idx + 1, len(lines)):
        lvl = heading_level(lines[j])
        if lvl is not None and lvl <= summary_lvl:     # next same/higher-level section ends the summary
            return "\n".join(lines[:j]).strip(), "\n".join(lines[j:]).strip()
    return report_text, ""                             # Summary is the last/only section


def md_len(block):
    return len(block["text"]) if block["type"] == "markdown" else 0


def pack(blocks):
    """Yield sublists of blocks, each <= MAX_BLOCKS_PER_MSG blocks and <= MAX_MD_CHARS markdown chars."""
    batch, md_total = [], 0
    for block in blocks:
        over_blocks = len(batch) + 1 > MAX_BLOCKS_PER_MSG
        over_chars  = md_total + md_len(block) > MAX_MD_CHARS
        if batch and (over_blocks or over_chars):
            yield batch
            batch, md_total = [], 0
        batch.append(block)
        md_total += md_len(block)
    if batch:
        yield batch


def md_blocks(text):
    return [{"type": "markdown", "text": c} for c in split_markdown(text)]


def post_message(blocks, fallback_text, thread_ts=None, max_retries=4):
    """POST one message via chat.postMessage, retrying on 429. Returns parsed JSON."""
    headers = {
        "Authorization": f"Bearer {SLACK_BOT_TOKEN}",
        "Content-Type": "application/json; charset=utf-8",
    }
    payload = {"channel": CHANNEL_ID, "text": fallback_text, "blocks": blocks}
    if thread_ts:
        payload["thread_ts"] = thread_ts
    for _ in range(max_retries):
        resp = requests.post(POST_URL, headers=headers, json=payload, timeout=10)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("Retry-After", "1")))
            continue
        resp.raise_for_status()
        return resp.json()
    raise RuntimeError("Slack rate limit: retries exhausted")


# ── Build and send ───────────────────────────────────────────────────────────
df = spark.table(reports_table)
games = [row.game_name for row in df.select("game_name").distinct().collect()]

for game in sorted(games):
    game_reports = df.filter(df.game_name == game).select("persona", "report_contents").collect()

    # Game title — H1 markdown heading (largest), bigger than the persona's H2
    game_heading = {"type": "markdown", "text": f"# {controller_emoji} {game}"}

    for idx, row in enumerate(game_reports):          # ← enumerate to know the first persona
        persona_label   = row.persona.replace("_", " ").title()
        persona_heading = {"type": "markdown", "text": f"## {person_emoji} {persona_label}"}

        report_text = row.report_contents.strip() if row.report_contents else "No report available."
        head, rest = split_at_summary(report_text)

        # Game H1 only on the first persona's post for this game; others show just the persona H2
        lead = [game_heading, persona_heading] if idx == 0 else [persona_heading]
        parent_batches = list(pack(lead + md_blocks(head)))
        
        parent = post_message(parent_batches[0], fallback_text=f"{game} — {persona_label}")
        if not parent.get("ok"):
            print(f"FAILED parent {game} / {persona_label}: {parent.get('error')} | {parent}")
            continue
        parent_ts = parent["ts"]

        # Threaded replies: any parent overflow (rare) + the rest of the report
        thread_batches = parent_batches[1:]
        if rest:
            thread_batches += list(pack(md_blocks(rest)))

        for n, batch in enumerate(thread_batches, 1):
            r = post_message(batch, fallback_text=f"{game} — {persona_label} (cont. {n})", thread_ts=parent_ts)
            if not r.get("ok"):
                print(f"  FAILED thread part {n} for {game} / {persona_label}: {r.get('error')}")
            time.sleep(1)

        print(f"{game} / {persona_label}: parent ts={parent_ts}, thread parts={len(thread_batches)}")
        time.sleep(1)